In [1]:
import sys
import os
import slurm_utils

HOME = os.environ['HOME']
BASECODE_PATH = os.path.join(HOME,'international-brain-lab/prior-localization/decoding')
if not BASECODE_PATH in sys.path:                                                                              
     sys.path.insert(0, BASECODE_PATH)

fb = open('slurm_decode_base.py','r')
filestr = fb.read()
fb.close()
exec(filestr)

fiteidstr = """
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
eid = sys.argv[1]
ADD_TO_SAVING_PATH = sys.argv[2]
print(eid)
if NULL_TYPE == 'impostor-session':
    IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA), 
                             '_'.join(['impostors',''])+'.pkl')
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)
else:
    impostordict = None
filenames = fit_eid(eid,sessdf,impostordict = impostordict)
print('saving to files:',filenames)
"""

ADD_TO_SAVING_PATH = 'alleid220414'
SUBDIRECTORY = dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA)
SLURM_DIRECTORY = os.path.join(GROUP_HOME,'bensonb','international-brain-lab/prior-localization/slurm/')
OUTPUT_PATH = os.path.join(GROUP_HOME,'bensonb/international-brain-lab/prior-localization/decoding/')
print(SUBDIRECTORY)

IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             SUBDIRECTORY, 
                             '_'.join(['impostors',''])+'.pkl')
if os.path.exists(IMPOSTOR_PATH):
    print('Impostor sessions already exist.')

decode_signcont_task_Lasso_r2_control_100_pseudo-session_align_goCue_times_timeWin_0_0_1_alleid220414


In [5]:
# make python file
py_file = os.path.join(OUTPUT_PATH, SUBDIRECTORY, '_'.join(['slurm_decode_eid',''])+'.py')
if not os.path.exists(os.path.join(OUTPUT_PATH, SUBDIRECTORY)):
    os.mkdir(os.path.join(OUTPUT_PATH, SUBDIRECTORY))
with open(py_file,'w') as fp:
    fp.write(filestr)
    fp.write(fiteidstr)
    
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
sessdf_eids = sessdf.index.unique(level='eid')

# sessdf_eids=np.unique(sessdf_eids)[30:32]
# sessdf_eids= ['dfd8e7df-dc51-4589-b6ca-7baccfeb94b4', '034e726f-b35f-41e0-8d6c-a22cc32391fb',
#             '7939711b-8b4d-4251-b698-b97c1eaa846e', 'fa704052-147e-46f6-b190-a65b837e605e',
#             '46794e05-3f6a-4d35-afb3-9165091a5a74', '1538493d-226a-46f7-b428-59ce5f43f0f9',
#             'b52182e7-39f6-4914-9717-136db589706e', '89f0d6ff-69f4-45bc-b89e-72868abb042a',
#             '3ce452b3-57b4-40c9-885d-1b814036e936', '2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e',
#             '4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b', 'd839491f-55d8-4cbe-a298-7839208ba12b',
#             'd2918f52-8280-43c0-924b-029b2317e62c', 'c99d53e6-c317-4c53-99ba-070b26673ac4',
#             'ecb5520d-1358-434c-95ec-93687ecd1396', '5386aba9-9b97-4557-abcd-abc2da66b863',
#             '4b00df29-3769-43be-bb40-128b1cba6d35', '3663d82b-f197-4e8b-b299-7b803a155b84',
#             '85dc2ebd-8aaf-46b0-9284-a197aee8b16f', '15f742e1-1043-45c9-9504-f1e8a53c1744']

# collect all target vectors for impostor sessions
if (not os.path.exists(IMPOSTOR_PATH)) and ('impostor-session' in SUBDIRECTORY):
    fimpostor = open(IMPOSTOR_PATH, 'wb')
    impostordict = {}
    n_eid = 0
    for eid in sessdf_eids:
        print(n_eid, len(sessdf_eids), eid)
        n_eid += 1
        impostordict[eid] = get_target_vector_eid(eid, sessdf)
    pickle.dump(impostordict, fimpostor)
    fimpostor.close()
elif 'impostor-session' in SUBDIRECTORY:
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)
else:
    impostordict = {}

fit_metadata['submitted_eids'] = sessdf_eids
fit_metadata['impostor_dictionary'] = impostordict
fn = os.path.join(OUTPUT_PATH,SUBDIRECTORY,'DATE_results')
fn = fn + '.parquet'
metadata_df = pd.Series({'filename': fn, **fit_metadata})
metadata_fn = '.'.join([fn.split('.')[0], 'metadata', 'pkl'])
metadata_fn = metadata_fn.replace('DATE',str(date.today()))
metadata_df.to_pickle(metadata_fn)
print('created metadata file and impostor sessions')

/home/users/bensonb/mambaforge/envs/iblenv/lib/python3.9/site-packages/one/api.py:1242: UserWarning: Newer cache tables require ONE version 1.10.0 or greater
  warnings.warn(f'Newer cache tables require ONE version {min_version} or greater')


created metadata file and impostor sessions


In [6]:
eid = '15f742e1-1043-45c9-9504-f1e8a53c1744'
one = ONE()  # mode='local'
atlas = AllenAtlas()

estimator = ESTIMATOR #(**ESTIMATOR_KWARGS)

subject = sessdf.xs(eid, level='eid').index[0]
subjeids = sessdf.xs(subject, level='subject').index.unique()
brainreg = dut.BrainRegions()
behavior_data = mut.load_session(eid, one=one)
pLeft_vec = np.array(behavior_data['probabilityLeft'])
try:
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, beh_data=behavior_data,
                              one=one)
except ValueError:
    print('Model not fit.')
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, one=one)

fvecs = [dut.compute_target(feature, subject, subjeids, eid, MODELFIT_PATH,
                          modeltype=MODEL, beh_data=behavior_data,
                          one=one) for feature in CONTROL_FEATURES]

try:
    trialsdf = bbone.load_trials_df(eid, one=one, addtl_types=['firstMovement_times'])
    if len(trialsdf) != len(tvec):
        raise IndexError
except IndexError:
    raise IndexError('Problem in the dimensions of dataframe of session')
trialsdf['react_times'] = trialsdf['firstMovement_times'] - trialsdf['goCue_times']

local md5 or size mismatch on dataset: cortexlab/Subjects/KS022/2019-12-09/001alf/_ibl_trials.response_times.npy


Downloading: /home/groups/sganguli/bensonb/Downloads/ONE/cache_dir/cortexlab/Subjects/KS022/2019-12-09/001/alf/_ibl_trials.response_times.e9ab5a29-9db4-4e8b-ac1f-ec82fdff65c0.npy Bytes: 6736


100%|█████████████████████████████████████████████████████████████████████████| 6736/6736 [00:00<00:00, 1137896.48it/s]
local md5 or size mismatch on dataset: cortexlab/Subjects/KS022/2019-12-09/001alf/_ibl_trials.response_times.npy


Downloading: /home/groups/sganguli/bensonb/Downloads/ONE/cache_dir/cortexlab/Subjects/KS022/2019-12-09/001/alf/_ibl_trials.response_times.e9ab5a29-9db4-4e8b-ac1f-ec82fdff65c0.npy Bytes: 6736


100%|█████████████████████████████████████████████████████████████████████████| 6736/6736 [00:00<00:00, 2523024.80it/s]


In [7]:
trialsdf.keys()

Index(['choice', 'probabilityLeft', 'feedbackType', 'feedback_times',
       'contrastLeft', 'contrastRight', 'goCue_times', 'stimOn_times',
       'firstMovement_times', 'trial_start', 'trial_end', 'react_times'],
      dtype='object')

In [12]:
def generate_shifts():
    out = np.arange(-10,10)
    np.random.shuffle(out)
    i = 0
    while i < len(out):
        yield out[i]
        i+=1
        
gs = generate_shifts()
for j in range(30):
    print(next(generate_shifts()))
    

-6
-1
-8
1
-1
7
4
5
7
-10
-5
-3
-5
-9
-1
6
-3
9
0
5
9
8
-7
-1
-3
9
-4
-10
9
1


In [6]:
# submit python file with eid inputs
for eid in sessdf_eids:
    slurm_utils.submit_python_file(py_file, 
                             eid, 
                             ADD_TO_SAVING_PATH = ADD_TO_SAVING_PATH,
                             n_gigs_ram = 8,
                             SLURM_DIRECTORY = SLURM_DIRECTORY,
                             SUBDIRECTORY = SUBDIRECTORY)
    

Submitted batch job 49629780
Submitted batch job 49629782
Submitted batch job 49629784
Submitted batch job 49629785
Submitted batch job 49629786
Submitted batch job 49629788
Submitted batch job 49629789
Submitted batch job 49629790
Submitted batch job 49629791
Submitted batch job 49629792
Submitted batch job 49629793
Submitted batch job 49629794
Submitted batch job 49629795
Submitted batch job 49629796
Submitted batch job 49629797
Submitted batch job 49629798
Submitted batch job 49629799
Submitted batch job 49629800
Submitted batch job 49629801
Submitted batch job 49629802
Submitted batch job 49629803
Submitted batch job 49629805
Submitted batch job 49629806
Submitted batch job 49629807
Submitted batch job 49629808
Submitted batch job 49629809
Submitted batch job 49629810
Submitted batch job 49629831
Submitted batch job 49629832
Submitted batch job 49629833
Submitted batch job 49629834
Submitted batch job 49629835
Submitted batch job 49629836
Submitted batch job 49629837
Submitted batc

Submitted batch job 49630090
Submitted batch job 49630091
Submitted batch job 49630092
Submitted batch job 49630094
Submitted batch job 49630095
Submitted batch job 49630096
Submitted batch job 49630097
Submitted batch job 49630098
Submitted batch job 49630099
Submitted batch job 49630100
Submitted batch job 49630101
Submitted batch job 49630102
Submitted batch job 49630103
Submitted batch job 49630104
Submitted batch job 49630105
Submitted batch job 49630106
Submitted batch job 49630107
Submitted batch job 49630108
Submitted batch job 49630109
Submitted batch job 49630110
Submitted batch job 49630120
Submitted batch job 49630121
Submitted batch job 49630122
Submitted batch job 49630123
Submitted batch job 49630124
Submitted batch job 49630125
Submitted batch job 49630126
Submitted batch job 49630127
Submitted batch job 49630128
Submitted batch job 49630129
Submitted batch job 49630130
Submitted batch job 49630131
Submitted batch job 49630132
Submitted batch job 49630133
Submitted batc

In [11]:

os.system("squeue -u $USER");
slurm_utils.get_decoding_output_files(SLURM_DIRECTORY = SLURM_DIRECTORY,
                                      SUBDIRECTORY = SUBDIRECTORY)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          49618365    normal interact  bensonb  R    6:20:33      1 sh02-01n08
          49618001    normal interact  bensonb  R    6:36:11      1 sh02-01n49


['/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_100_pseudo-session_align_goCue_times_timeWin_0_0_1_alleid220414/SWC_038/1e45d992-c356-40e1-9be1-a506d944896f/probe01/2022-04-14_ACB.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_100_pseudo-session_align_goCue_times_timeWin_0_0_1_alleid220414/SWC_038/1e45d992-c356-40e1-9be1-a506d944896f/probe01/2022-04-14_CP.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_100_pseudo-session_align_goCue_times_timeWin_0_0_1_alleid220414/SWC_038/1e45d992-c356-40e1-9be1-a506d944896f/probe01/2022-04-14_MOs.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_signcont_task_Lasso_r2_control_100_pseudo-session_align_goCue_times_timeWin_0_0_1_alleid220414/SWC_038/1e45d992-c356-40e1-9be1-a50

In [12]:
#SUBDIRECTORY = 'decode_pLeft_task_Logistic_accuracy_control_10_impostor-session_align_goCue_times_timeWin_-0_4_-0_1_testnulltype1'

slurm_utils.gather_save_outputs(SUBDIRECTORY, SLURM_DIRECTORY, OUTPUT_PATH)